## Find new colin events - split corp nums into batches

In [ ]:
import re
from pathlib import Path
from math import ceil

# ====== CONFIG ======
MODE = "identifiers"
# MODE = "all_after_cutoff"

# In SQL, Oracle type constructors are limited to 999 parameters.
# So keep this <= 999 to avoid ORA-00939 in sys.odcivarchar2list(...).
BATCH_SIZE = 999

# How many corp nums to print per *line* inside sys.odcivarchar2list(...)
# (this is just formatting; set 500-999 as you requested)
VALUES_PER_LINE = 999

CUTOFF = "TIMESTAMP '2026-03-23 06:00:00'"  # or ":cutoff_ts"
OUTPUT_FILE = "generated/check_events.sql"

IDENTIFIERS = r"""
'1111111', '2222222', '3333333'
-- paste your full list here exactly as provided (single quoted, comma-separated)
"""
# ====================

VALID_MODES = {"identifiers", "all_after_cutoff"}
# Match the supported full extract corp types from transfer_cprd_corps.sql
# and subset_transfer_chunk.sql.
FULL_EXTRACT_CORP_TYPE_FILTER = "('BC', 'C', 'ULC', 'CUL', 'CC', 'CCC', 'QA', 'QB', 'QC', 'QD', 'QE')"

def parse_ids(s: str) -> list[str]:
    # Extract values inside single quotes; preserves leading zeros like '0766566'
    ids = re.findall(r"'([^']*)'", s)

    # De-dupe while preserving order (optional; remove if you want duplicates preserved)
    seen = set()
    out: list[str] = []
    for x in ids:
        if x and x not in seen:
            seen.add(x)
            out.append(x)
    return out

def format_as_odcivarchar2list_args(items: list[str], per_line: int) -> str:
    """
    Returns a multi-line string of quoted items, comma-separated, with per_line items per line.
    Example (per_line=3):
        'a','b','c',
        'd','e'
    """
    lines: list[str] = []
    for i in range(0, len(items), per_line):
        chunk = items[i:i + per_line]
        lines.append("    " + ",".join(f"'{v}'" for v in chunk))
    return ",\n".join(lines)

def make_batch_cte(cte_name: str, items: list[str]) -> str:
    args_sql = format_as_odcivarchar2list_args(items, VALUES_PER_LINE)
    lines = [
        f"{cte_name} AS (",
        "  SELECT column_value AS corp_num",
        "  FROM TABLE(sys.odcivarchar2list(",
        args_sql,
        "  ))",
        ")",
    ]
    return "\n".join(lines)

def build_identifier_scope(items: list[str]) -> tuple[list[str], int, list[list[str]]]:
    n = len(items)
    num_batches = ceil(n / BATCH_SIZE)
    batches = [
        items[i * BATCH_SIZE : (i + 1) * BATCH_SIZE]
        for i in range(num_batches)
    ]

    cte_blocks = []
    batch_names = []
    for idx, batch in enumerate(batches, start=1):
        name = f"corp_list_{idx:03d}"
        batch_names.append(name)
        cte_blocks.append(make_batch_cte(name, batch))

    corp_union_lines = ["corp_list AS ("]
    corp_union_lines.append(f"  SELECT corp_num FROM {batch_names[0]}")
    for name in batch_names[1:]:
        corp_union_lines.append(f"  UNION ALL SELECT corp_num FROM {name}")
    corp_union_lines.append(")")

    return [*cte_blocks, "\n".join(corp_union_lines)], n, batches

def build_scope(mode: str, identifiers: str) -> tuple[list[str], str, dict]:
    if mode not in VALID_MODES:
        raise SystemExit(f"Unsupported MODE={mode!r}. Choose one of: {sorted(VALID_MODES)}")

    if mode == "identifiers":
        corp_nums = parse_ids(identifiers)
        if not corp_nums:
            raise SystemExit("No corp nums found. Paste the quoted list into IDENTIFIERS.")

        scope_ctes, n, batches = build_identifier_scope(corp_nums)
        join_sql = "  JOIN corp_list c\n    ON c.corp_num = e.corp_num\n"
        return scope_ctes, join_sql, {
            "corp_count": n,
            "batch_count": len(batches),
            "batch_sizes": [len(batch) for batch in batches],
            "uses_identifier_scope": True,
        }

    ignored_ids = parse_ids(identifiers)
    if ignored_ids:
        print(f"MODE={mode}; ignoring {len(ignored_ids)} identifiers from IDENTIFIERS.")

    join_sql = (
        "  JOIN corporation c\n"
        "    ON c.corp_num = e.corp_num\n"
        f"   AND c.corp_typ_cd IN {FULL_EXTRACT_CORP_TYPE_FILTER}\n"
    )

    return [], join_sql, {
        "corp_count": 0,
        "batch_count": 0,
        "batch_sizes": [],
        "uses_identifier_scope": False,
    }

scope_ctes, scope_join_sql, meta = build_scope(MODE, IDENTIFIERS)

latest_event_sql = f"""latest_event AS (
  SELECT e.event_id,
         e.corp_num,
         e.event_typ_cd,
         e.event_timestmp,
         e.trigger_dts,
         ROW_NUMBER() OVER (
           PARTITION BY e.corp_num
           ORDER BY e.event_timestmp DESC, e.event_id DESC
         ) AS rn
  FROM event e
{scope_join_sql}  WHERE e.event_timestmp > {CUTOFF}
)"""

with_parts = [*scope_ctes, latest_event_sql]
with_clause = ",\n".join(with_parts)

sql = f"""\
WITH
{with_clause}
SELECT le.EVENT_ID,
       le.corp_num,
       le.event_typ_cd,
       le.event_timestmp,
       le.trigger_dts,
       f.FILING_TYP_CD
FROM latest_event le
left join filing f on le.EVENT_ID = f.EVENT_ID
WHERE rn = 1
ORDER BY le.event_timestmp DESC, le.EVENT_ID DESC;
"""

Path(OUTPUT_FILE).write_text(sql, encoding="utf-8")

print(f"Wrote {OUTPUT_FILE}")
print(f"Mode: {MODE}")
if meta["uses_identifier_scope"]:
    print(f"Total corp nums: {meta['corp_count']}")
    print(f"Batch size (per odcivarchar2list): {BATCH_SIZE}")
    print(f"Values per line (formatting only): {VALUES_PER_LINE}")
    print(f"Batches: {meta['batch_count']} -> {meta['batch_sizes']}")
else:
    print("Identifier scope: disabled (querying all supported full-extract corp types after cutoff)")
